# SBC extractor: QLoRA fine-tune of Qwen3-4B (Kaggle T4)

**Notebook settings:** Accelerator **GPU T4** · Internet **on** · add the `sbc-sft` dataset
(uploaded from `dist/kaggle-sbc-sft/`). Background runs: *Save Version → Save & Run All*.

Steps: dry run (5 steps) → full run (~2 epochs) → gold-set predictions with vLLM → download `results.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls /kaggle/input

In [ ]:
# Adjust if `ls /kaggle/input` shows a different folder name
DATA = "/kaggle/input/sbc-sft"
OUT = "/kaggle/working/qwen3-4b-sbc-lora"
!ls $DATA

In [ ]:
!pip install -q -U "trl>=0.20" peft bitsandbytes liger-kernel datasets accelerate

## 1. Dry run: 5 optimizer steps
Checks memory and that the loss is a real number (not `nan`) before committing GPU hours.
Expect peak memory around 10–14 GiB on a T4.

In [ ]:
!python $DATA/train_qlora.py --data-dir $DATA --output-dir /kaggle/working/dryrun --max-steps 5 --eval-docs 2 --eval-steps 5

## 2. Full run
~94 optimizer steps (2 epochs × 376 examples ÷ 8). Watch `loss` and `eval_loss` every 20 steps.

In [ ]:
!python $DATA/train_qlora.py --data-dir $DATA --output-dir $OUT
!cat $OUT/final/log_history.json | head -80

## 3. Gold-set predictions with vLLM + LoRA
vLLM may pin its own torch version, so install it **after** training. If the import fails,
restart the session and run only this section.

In [ ]:
!pip install -q vllm
!python $DATA/eval_gold_vllm.py --data-dir $DATA --adapter $OUT/final --run qwen3-4b-lora-r16
!python $DATA/eval_gold_vllm.py --data-dir $DATA --run qwen3-4b-base-student-prompt

In [ ]:
!cd /kaggle/working && zip -qr results.zip qwen3-4b-sbc-lora/final predictions && ls -lh results.zip

Download `results.zip` from the Output panel, unzip `predictions/*` into `eval/predictions/`
in the repo, and score:

```bash
uv run python eval/score.py eval/predictions/qwen3-4b-lora-r16
```